In [0]:
# DO NOT MODIFY


In [0]:
import importlib.metadata as md
import subprocess, sys

try:
    before = md.version("databricks-sdk")
except md.PackageNotFoundError:
    before = None

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "databricks-sdk>=0.118.0"])
after = md.version("databricks-sdk")
print(f"databricks-sdk: {before} -> {after}  (changed={before != after})")

if before != after:
    print("Version changed — restarting Python to load the new SDK...")
    dbutils.library.restartPython()


In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# ── Config ─────────────────────────────────────────────────────────────────
PROJECT_ID = "sentenel-tech-summit-27"
BRANCH_ID = "production"
DB_NAME = "sentenel_tech_summit_27"  # friendly Postgres database name
DB_ID = "db-" + DB_NAME.replace("_", "-")  # resource ID (hyphens only)

# ── 1. Verify project exists ───────────────────────────────────────────────
project = w.postgres.get_project(name=f"projects/{PROJECT_ID}")
print(f"[setup-db] project: {project.name}")

# ── 2. Ensure branch exists ────────────────────────────────────────────────
branch_path = f"projects/{PROJECT_ID}/branches/{BRANCH_ID}"
try:
    branch = w.postgres.get_branch(name=branch_path)
    print(f"[setup-db] branch: {branch.name} (state={branch.status.current_state})")
except Exception:
    from databricks.sdk.service.postgres import Branch, BranchSpec
    print(f"[setup-db] creating branch '{BRANCH_ID}'...")
    branch = w.postgres.create_branch(
        parent=f"projects/{PROJECT_ID}",
        branch=Branch(spec=BranchSpec(no_expiry=True)),
        branch_id=BRANCH_ID,
    ).wait()
    print(f"[setup-db] branch created: {branch.name}")

# ── 3. Ensure endpoint exists ──────────────────────────────────────────────
endpoint_path = f"{branch_path}/endpoints/primary"
try:
    ep = w.postgres.get_endpoint(name=endpoint_path)
    print(f"[setup-db] endpoint: {ep.name}")
except Exception:
    from databricks.sdk.service.postgres import Endpoint, EndpointSpec, EndpointType
    print("[setup-db] creating primary endpoint...")
    ep = w.postgres.create_endpoint(
        parent=branch_path,
        endpoint=Endpoint(spec=EndpointSpec(
            endpoint_type=EndpointType.ENDPOINT_TYPE_READ_WRITE,
            autoscaling_limit_min_cu=0.5,
            autoscaling_limit_max_cu=2,
        )),
        endpoint_id="primary",
    ).wait()
    print(f"[setup-db] endpoint created: {ep.name}")

# ── 4. Ensure database exists ──────────────────────────────────────────────
db_path = f"{branch_path}/databases/{DB_ID}"
try:
    db = w.postgres.get_database(name=db_path)
    print(f"[setup-db] database '{DB_NAME}' (id={DB_ID}): exists")
except Exception:
    # Get first role to use as owner
    roles = list(w.postgres.list_roles(parent=branch_path))
    owner_role = roles[0].name if roles else None
    if not owner_role:
        raise RuntimeError("No roles found on branch — cannot create database")
    from databricks.sdk.service.postgres import Database, DatabaseDatabaseSpec
    print(f"[setup-db] creating database '{DB_NAME}' (owner={owner_role})...")
    db = w.postgres.create_database(
        parent=branch_path,
        database=Database(spec=DatabaseDatabaseSpec(
            postgres_database=DB_NAME,
            role=owner_role,
        )),
        database_id=DB_ID,
    ).wait()
    print(f"[setup-db] database '{DB_NAME}' (id={DB_ID}): created")

# ── 5. Print connection details ────────────────────────────────────────────
ep = w.postgres.get_endpoint(name=endpoint_path)
pg_host = ep.status.hosts.host

print(f"""
[setup-db] done. Use these values for your .env / app config:

  LAKEBASE_PROJECT_ID={PROJECT_ID}
  LAKEBASE_ENDPOINT={branch_path}/endpoints/primary
  PGHOST={pg_host}
  PGDATABASE={DB_NAME}

  # Database resource path (for the App's bundle postgres binding):
  {branch_path}/databases/{DB_ID}
""")